In [4]:
from pyngrok import ngrok
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State
import threading, time, os, base64, io

NGROK_TOKEN = "3IRcJjmxbKsfuy9qfLBbgEnTT17_38idA3RNR1WCKVEkNgmgU"
ngrok.set_auth_token(NGROK_TOKEN)
os.system("pkill -9 -f ngrok")
ngrok.kill()
time.sleep(1)
try:
    for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
except: pass

DF_GLOBAL = None

C_ORANGE = "#FF6B2E"; C_HEADER = "#FF7A3D"; C_PEACH = "#FDDCC9"
ORANGE_GRADIENT = ["#E05A20", "#FF6B2E", "#FF8C42", "#FFA94D", "#FFC96B"]
C_GRAY = "#D0D0D0"
COHORT_COLORS = {"Arabic":"#FFC96B","English":"#FF8C42"}
SECTOR_COLORS = px.colors.qualitative.Bold

def get_colors(labels): return [C_GRAY if str(l).lower() in ["not specified"] else ORANGE_GRADIENT[i % len(ORANGE_GRADIENT)] for i,l in enumerate(labels)]
def get_top_insight(series):
    if len(series)==0: return "No Data", 0
    try: filtered = series[~series.index.str.lower().isin(["not specified","nan"])]
    except: filtered = series
    if len(filtered)==0: filtered = series
    return filtered.idxmax(), filtered.max()

app = Dash(__name__)
btn_w = {'background':'white','color':'black','border':f'2px solid {C_PEACH}','borderRadius':'12px','padding':'10px','fontWeight':'700','cursor':'pointer','fontSize':'11px'}
btn_a = {'background':C_ORANGE,'color':'white','border':'none','borderRadius':'12px','padding':'10px','fontWeight':'700','cursor':'pointer','fontSize':'11px'}

app.layout = html.Div(style={'fontFamily':'Segoe UI','display':'flex','height':'100vh'}, children=[
    html.Div(style={'width':'285px','background':'white','borderRight':f'3px solid {C_PEACH}','padding':'16px','display':'flex','flexDirection':'column','gap':'8px','overflowY':'auto'}, children=[
        html.Div("Mashroo3i", style={'color':C_ORANGE,'fontSize':'26px','fontWeight':'800','textAlign':'center'}),
        dcc.Upload(id='upload-csv', children=html.Div(['📤 Upload'], style={'textAlign':'center','fontWeight':'800','fontSize':'14px'}), style={'width':'100%','height':'45px','lineHeight':'45px','borderWidth':'2px','borderStyle':'dashed','borderColor':C_ORANGE,'borderRadius':'12px','background':C_PEACH,'cursor':'pointer'}, multiple=False),
        html.Div(id='upload-status'),
        html.Hr(),
        html.Button("Page 1 - Overview", id='btn-p1', n_clicks=0, style=btn_a),
        html.Button("Page 2 - Applicant Profile", id='btn-p2', n_clicks=0, style=btn_w),
        html.Button("Page 3 - Sectors & Type", id='btn-p3', n_clicks=0, style=btn_w),
        html.Button("Page 4 - Cohort Comparison", id='btn-p4', n_clicks=0, style=btn_w),
        html.Hr(),
        dcc.Dropdown(id='f-year', options=[], multi=True, placeholder="All Years"),
        dcc.Dropdown(id='f-cohort', options=[{'label':c,'value':c} for c in ['Arabic','English']], multi=True, placeholder="Arabic / English"),
        dcc.Dropdown(id='f-outcome', options=[], multi=True, placeholder="All Outcomes"),
        dcc.Dropdown(id='f-sector', options=[], multi=True, placeholder="All Sectors"),
        dcc.Dropdown(id='f-type', options=[], multi=True, placeholder="Individual or Team"),
    ]),
    html.Div(style={'flex':'1','padding':'20px','overflowY':'auto','background':C_PEACH}, children=[
        dcc.Loading(type="circle", color=C_ORANGE, children=html.Div(id='page-content'))
    ]),
    dcc.Store(id='current-page', data='page1'),
])

@app.callback(Output('current-page','data'), Output('btn-p1','style'), Output('btn-p2','style'), Output('btn-p3','style'), Output('btn-p4','style'),
              Input('btn-p1','n_clicks'), Input('btn-p2','n_clicks'), Input('btn-p3','n_clicks'), Input('btn-p4','n_clicks'))
def switch_page(b1,b2,b3,b4):
    from dash import ctx
    if not ctx.triggered: return 'page1', btn_a, btn_w, btn_w, btn_w
    m = {'btn-p1':'page1','btn-p2':'page2','btn-p3':'page3','btn-p4':'page4'}
    page = m.get(ctx.triggered_id, 'page1')
    return page, btn_a if page=='page1' else btn_w, btn_a if page=='page2' else btn_w, btn_a if page=='page3' else btn_w, btn_a if page=='page4' else btn_w

@app.callback(Output('upload-status','children'),
              Output('f-year','options'), Output('f-outcome','options'), Output('f-sector','options'), Output('f-type','options'),
              Input('upload-csv','contents'), State('upload-csv','filename'))
def handle_upload(contents, filename):
    global DF_GLOBAL
    if contents is None: return "", [], [], [], []
    _, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)
    df_up = pd.read_csv(io.BytesIO(decoded), encoding='utf-8-sig') if filename.lower().endswith('.csv') else pd.read_excel(io.BytesIO(decoded))
    keep = ['year','cohort','Sector','outcome_clean','Business Stage','Age Group','applicant_type','team_member_count','employment_status','has_commercial_registration','education','major','nationality']
    DF_GLOBAL = df_up[[c for c in keep if c in df_up.columns]].copy()
    msg = html.Div(f"✅ {len(DF_GLOBAL)} rows", style={'background':'white','borderRadius':'8px','padding':'8px','border':f'1px solid {C_ORANGE}','fontSize':'11px','fontWeight':'700','textAlign':'center','color':'green'})
    year_opts = [{'label':str(int(y)), 'value':int(y)} for y in sorted(DF_GLOBAL['year'].dropna().unique())]
    outcome_opts = [{'label':str(o), 'value':o} for o in DF_GLOBAL['outcome_clean'].dropna().unique()]
    sector_opts = [{'label':str(s), 'value':s} for s in DF_GLOBAL['Sector'].dropna().unique()]
    type_opts = [{'label':str(t), 'value':t} for t in DF_GLOBAL['applicant_type'].dropna().unique()]
    return msg, year_opts, outcome_opts, sector_opts, type_opts

# === هنا التعديل المهم: اضفنا upload-status كـ Input عشان الرسم يطلع مباشرة بعد الابلود ===
@app.callback(Output('page-content','children'),
              Input('current-page','data'),
              Input('upload-status','children'), # <-- هذا يخلي الرسم يتكون سيده بعد الابلود
              Input('f-year','value'), Input('f-cohort','value'), Input('f-outcome','value'), Input('f-sector','value'), Input('f-type','value'))
def update_page(page, _upload_trigger, years, cohorts, outcomes, sectors, types):
    global DF_GLOBAL
    if DF_GLOBAL is None:
        return [html.Div(style={'background':'white','borderRadius':'20px','padding':'80px','textAlign':'center'}, children=[html.Div("📤", style={'fontSize':'60px'}), html.Div("Upload your file to start", style={'fontSize':'14px','fontWeight':'700','color':'#999','marginTop':'15px'})])]
    dff = DF_GLOBAL
    if years: dff = dff[dff['year'].isin(years)]
    if cohorts: dff = dff[dff['cohort'].isin(cohorts)]
    if outcomes: dff = dff[dff['outcome_clean'].isin(outcomes)]
    if sectors: dff = dff[dff['Sector'].isin(sectors)]
    if types: dff = dff[dff['applicant_type'].isin(types)]
    if len(dff)==0:
        return [html.Div("No data for this filter", style={'background':'white','borderRadius':'20px','padding':'40px','textAlign':'center','fontWeight':'800'})]

    total=len(dff); accepted=len(dff[dff['outcome_clean']=='Accepted']); rate=round(accepted/total*100,1) if total else 0
    bah_rate=round(len(dff[dff['nationality'].astype(str).str.contains('bahrain', case=False, na=False)])/total*100,1) if total else 0

    kpis = html.Div(style={'display':'flex','gap':'12px','marginBottom':'16px','flexWrap':'wrap'}, children=[
        html.Div(style={'background':'white','borderRadius':'20px','padding':'18px','textAlign':'center','flex':'1','minWidth':'130px'}, children=[html.Div("👥"), html.Div(f"{total}", style={'fontSize':'30px','fontWeight':'800'}), html.Div("Total Applicants", style={'color':C_ORANGE,'fontWeight':'700','fontSize':'11px'})]),
        html.Div(style={'background':'white','borderRadius':'20px','padding':'18px','textAlign':'center','flex':'1','minWidth':'130px'}, children=[html.Div("✅"), html.Div(f"{accepted}", style={'fontSize':'30px','fontWeight':'800'}), html.Div("Accepted", style={'color':C_ORANGE,'fontWeight':'700','fontSize':'11px'})]),
        html.Div(style={'background':'white','borderRadius':'20px','padding':'18px','textAlign':'center','flex':'1','minWidth':'130px'}, children=[html.Div("📈"), html.Div(f"{rate}%", style={'fontSize':'30px','fontWeight':'800'}), html.Div("Acceptance Rate", style={'color':C_ORANGE,'fontWeight':'700','fontSize':'11px'})]),
        html.Div(style={'background':'white','borderRadius':'20px','padding':'18px','textAlign':'center','flex':'1','minWidth':'130px'}, children=[html.Img(src="https://flagcdn.com/w80/bh.png", style={'width':'32px','height':'20px','borderRadius':'3px'}), html.Div(f"{bah_rate}%", style={'fontSize':'30px','fontWeight':'800'}), html.Div("Bahraini Nationals", style={'color':C_ORANGE,'fontWeight':'700','fontSize':'11px'})]),
    ])
    def card(title, fig):
        return html.Div(style={'background':'white','borderRadius':'20px','overflow':'hidden','flex':'1','minWidth':'360px'}, children=[html.Div(title, style={'background':C_HEADER,'color':'white','padding':'14px 12px','textAlign':'center','fontWeight':'800','fontSize':'13px'}), dcc.Graph(figure=fig, config={'displayModeBar':False})])

    if page == 'page1':
        df_yc = dff.groupby(['year','cohort']).size().reset_index(name='Total')
        yearly_total = df_yc.groupby('year')['Total'].sum()
        top_year, top_year_val = get_top_insight(yearly_total)
        fig_y = px.bar(df_yc, x='year', y='Total', color='cohort', barmode='stack', color_discrete_map=COHORT_COLORS, text='Total', category_orders={'cohort':['Arabic','English']})
        fig_y.update_traces(marker=dict(cornerradius=12), textposition='outside')
        fig_y.update_layout(height=380, margin=dict(l=20,r=20,t=70,b=50), legend=dict(orientation="h", y=1.22, x=0.5, xanchor="center"), xaxis=dict(title="", showgrid=False, showline=False, ticks="", tickfont=dict(size=13, color="#333")), yaxis=dict(visible=False, range=[0, yearly_total.max()*1.5]))
        cnt_type = dff['applicant_type'].value_counts()
        top_type, top_type_val = get_top_insight(cnt_type)
        fig_type = go.Figure(go.Bar(x=cnt_type.index, y=cnt_type.values, marker=dict(color=get_colors(cnt_type.index), cornerradius=14), text=cnt_type.values, textposition='outside'))
        fig_type.update_layout(height=360, margin=dict(l=20,r=80,t=10,b=40), yaxis=dict(visible=False, range=[0, cnt_type.max()*1.6]), xaxis=dict(title="", visible=True, showgrid=False))
        cnt_team = dff['team_member_count'].value_counts().head(5)
        fig_team = go.Figure(go.Bar(x=["One","Two","Three","Four","Five"][:len(cnt_team)], y=cnt_team.values, marker=dict(color=ORANGE_GRADIENT[:len(cnt_team)], cornerradius=14), text=cnt_team.values, textposition='outside'))
        fig_team.update_layout(height=360, margin=dict(l=20,r=80,t=10,b=40), yaxis=dict(visible=False, range=[0, cnt_team.max()*1.6]), xaxis=dict(title="", visible=True, showgrid=False))
        return [kpis, html.Div(style={'background':'white','borderRadius':'20px','overflow':'hidden','marginBottom':'16px'}, children=[html.Div(f"Peak Year {int(top_year)} Leads with {int(top_year_val)} Applications", style={'background':C_HEADER,'color':'white','padding':'14px','textAlign':'center','fontWeight':'800'}), dcc.Graph(figure=fig_y)]), html.Div(style={'display':'flex','gap':'14px','flexWrap':'wrap'}, children=[card(f"{top_type} Leads - {int(top_type_val)} Applicants", fig_type), card("Small Teams Most Common", fig_team)])]
    elif page == 'page2':
        def make_h(col):
            cnt = dff[col].value_counts()
            cnt_plot = cnt.sort_values(ascending=True).tail(6)
            fig = go.Figure(go.Bar(y=cnt_plot.index, x=cnt_plot.values, orientation='h', marker=dict(color=get_colors(cnt_plot.index), cornerradius=14), text=cnt_plot.values, textposition='outside'))
            fig.update_layout(height=400, margin=dict(l=30,r=90,t=10,b=30), xaxis=dict(title="", visible=False, range=[0, cnt.max()*1.6]), yaxis=dict(title="", visible=True, showgrid=False))
            return fig, cnt, *get_top_insight(cnt)
        fig_stage, cnt_stage, top_stage, top_stage_val = make_h('Business Stage')
        fig_age, cnt_age, top_age, top_age_val = make_h('Age Group')
        cnt_out = dff['outcome_clean'].value_counts()
        top_out, top_out_val = get_top_insight(cnt_out)
        fig_out = go.Figure(go.Bar(x=cnt_out.index, y=cnt_out.values, marker=dict(color=get_colors(cnt_out.index), cornerradius=14), text=cnt_out.values, textposition='outside'))
        fig_out.update_layout(height=400, margin=dict(l=20,r=90,t=10,b=40), xaxis=dict(title="", visible=True, showgrid=False), yaxis=dict(title="", visible=False, range=[0, cnt_out.max()*1.6]))
        cnt_cr = dff['has_commercial_registration'].value_counts()
        top_cr, top_cr_val = get_top_insight(cnt_cr)
        fig_cr = go.Figure(go.Bar(x=cnt_cr.index, y=cnt_cr.values, marker=dict(color=get_colors(cnt_cr.index), cornerradius=14), text=cnt_cr.values, textposition='outside'))
        fig_cr.update_layout(height=400, margin=dict(l=20,r=90,t=10,b=40), xaxis=dict(title="", visible=True, showgrid=False), yaxis=dict(title="", visible=False, range=[0, cnt_cr.max()*1.6]))
        fig_emp, cnt_emp, top_emp, top_emp_val = make_h('employment_status')
        fig_edu, cnt_edu, top_edu, top_edu_val = make_h('education')
        fig_major, cnt_major, top_major, top_major_val = make_h('major')
        return [kpis, html.Div(style={'display':'flex','gap':'14px','marginBottom':'14px','flexWrap':'wrap'}, children=[card(f"{top_stage} Stage - {int(top_stage_val)}", fig_stage), card(f"{top_age} Leads - {int(top_age_val)}", fig_age), card(f"{top_out} - {int(top_out_val)}", fig_out)]), html.Div(style={'display':'flex','gap':'14px','marginBottom':'14px','flexWrap':'wrap'}, children=[card(f"{top_cr} Leads CR - {int(top_cr_val)}", fig_cr), card(f"{top_emp} - {int(top_emp_val)}", fig_emp), card(f"{top_edu} - {int(top_edu_val)}", fig_edu)]), html.Div(style={'display':'flex','gap':'14px','flexWrap':'wrap'}, children=[card(f"{top_major} - {int(top_major_val)}", fig_major)])]
    elif page == 'page3':
        cnt_sec = dff['Sector'].value_counts()
        top_sec, top_sec_val = get_top_insight(cnt_sec)
        fig_sec = go.Figure(go.Bar(y=cnt_sec.head(5).sort_values().index, x=cnt_sec.head(5).sort_values().values, orientation='h', marker=dict(color=get_colors(cnt_sec.head(5).sort_values().index), cornerradius=14), text=cnt_sec.head(5).sort_values().values, textposition='outside'))
        fig_sec.update_layout(height=380, margin=dict(l=30,r=90,t=10,b=30), xaxis=dict(title="", visible=False, range=[0, cnt_sec.max()*1.6]), yaxis=dict(title="", visible=True, showgrid=False))
        df_type_out = dff.groupby(['applicant_type','outcome_clean']).size().reset_index(name='Total')
        fig_type_out = px.bar(df_type_out, y='applicant_type', x='Total', color='outcome_clean', orientation='h', barmode='stack', color_discrete_map={"Accepted":"#E05A20","Rejected":"#FF8C42","Not Specified":C_GRAY}, text='Total')
        fig_type_out.update_traces(marker=dict(cornerradius=14), textposition='inside')
        fig_type_out.update_layout(height=380, margin=dict(l=20,r=20,t=70,b=10), legend=dict(orientation="h", y=1.25, x=0.5, xanchor="center"), xaxis=dict(title="", visible=False), yaxis=dict(title="", visible=True, showgrid=False))
        cnt_y_type = dff.groupby(['year','applicant_type']).size().reset_index(name='Total')
        fig_y_type = px.bar(cnt_y_type, x='year', y='Total', color='applicant_type', barmode='group', color_discrete_map={"Individual":"#E05A20","Team":"#FF8C42"}, text='Total')
        fig_y_type.update_traces(marker=dict(cornerradius=12), textposition='outside')
        fig_y_type.update_layout(height=380, margin=dict(l=10,r=80,t=70,b=40), legend=dict(orientation="h", y=1.25, x=0.5, xanchor="center"), xaxis=dict(title="", visible=True, showgrid=False), yaxis=dict(title="", visible=False, range=[0, cnt_y_type.groupby('year')['Total'].sum().max()*1.6]), bargap=0.4, bargroupgap=0.3)
        return [kpis, html.Div(style={'background':'white','borderRadius':'20px','overflow':'hidden','marginBottom':'14px'}, children=[html.Div("Teams Achieve Higher Acceptance", style={'background':C_HEADER,'color':'white','padding':'14px','textAlign':'center','fontWeight':'800'}), dcc.Graph(figure=fig_type_out)]), html.Div(style={'display':'flex','gap':'14px','flexWrap':'wrap'}, children=[card(f"{top_sec} Leads - {int(top_sec_val)}", fig_sec), html.Div(style={'background':'white','borderRadius':'20px','overflow':'hidden','flex':'1','minWidth':'360px'}, children=[html.Div("Team Applications Grow Year Over Year", style={'background':C_HEADER,'color':'white','padding':'14px','textAlign':'center','fontWeight':'800'}), dcc.Graph(figure=fig_y_type)])])]
    else:
        df_rate = dff.groupby('cohort')['outcome_clean'].value_counts(normalize=True).unstack(fill_value=0).reset_index()
        df_rate['Acceptance Rate %'] = (df_rate['Accepted']*100).round(1) if 'Accepted' in df_rate.columns else 0
        df_rate = df_rate.sort_values('Acceptance Rate %', ascending=False)
        fig_rate = px.bar(df_rate, x='cohort', y='Acceptance Rate %', color='cohort', color_discrete_map=COHORT_COLORS, text='Acceptance Rate %', category_orders={'cohort':['Arabic','English']})
        fig_rate.update_traces(marker=dict(cornerradius=12), textposition='outside')
        fig_rate.update_layout(height=400, showlegend=False, margin=dict(l=20,r=90,t=10,b=40), xaxis=dict(title="", visible=True), yaxis=dict(title="", visible=False, range=[0, df_rate['Acceptance Rate %'].max()*1.5]))
        cnt_cohort = dff['cohort'].value_counts()
        top_cohort, top_cohort_val = get_top_insight(cnt_cohort)
        fig_cohort = go.Figure(go.Bar(y=cnt_cohort.sort_values().index, x=cnt_cohort.sort_values().values, orientation='h', marker=dict(color=[COHORT_COLORS.get(str(c), "#FF6B2E") for c in cnt_cohort.sort_values().index], cornerradius=14), text=cnt_cohort.sort_values().values, textposition='outside'))
        fig_cohort.update_layout(height=400, margin=dict(l=30,r=90,t=10,b=30), xaxis=dict(title="", visible=False, range=[0, cnt_cohort.max()*1.6]), yaxis=dict(title="", visible=True, showgrid=False))
        df_sec_cohort = dff.groupby(['cohort','Sector']).size().reset_index(name='Total').sort_values('Total', ascending=False).groupby('cohort').head(3)
        fig_sec_cohort = px.bar(df_sec_cohort, x='cohort', y='Total', color='Sector', barmode='group', text='Total', color_discrete_sequence=SECTOR_COLORS, category_orders={'cohort':['Arabic','English']})
        fig_sec_cohort.update_traces(marker=dict(cornerradius=10), textposition='outside')
        fig_sec_cohort.update_layout(height=450, bargap=0.45, bargroupgap=0.35, legend=dict(orientation="h", y=1.35, x=0.5, xanchor="center"), margin=dict(l=20,r=90,t=110,b=30), xaxis=dict(title="", visible=True, showgrid=False), yaxis=dict(title="", visible=False, range=[0, df_sec_cohort['Total'].max()*2.2]))
        df_type_cohort = dff.groupby(['cohort','applicant_type']).size().reset_index(name='Total')
        fig_type_cohort = px.bar(df_type_cohort, x='cohort', y='Total', color='applicant_type', barmode='group', text='Total', color_discrete_map={"Individual":"#E05A20","Team":"#FF8C42"}, category_orders={'cohort':['Arabic','English']})
        fig_type_cohort.update_traces(marker=dict(cornerradius=10), textposition='outside')
        fig_type_cohort.update_layout(height=450, bargap=0.45, bargroupgap=0.35, legend=dict(orientation="h", y=1.35, x=0.5, xanchor="center"), margin=dict(l=20,r=90,t=110,b=30), xaxis=dict(title="", visible=True, showgrid=False), yaxis=dict(title="", visible=False, range=[0, df_type_cohort['Total'].max()*2.2]))
        return [kpis, html.Div(style={'display':'flex','gap':'14px','marginBottom':'14px','flexWrap':'wrap'}, children=[card(f"{df_rate.iloc[0]['cohort']} Highest Acceptance - {df_rate.iloc[0]['Acceptance Rate %']}% Rate" if len(df_rate)>0 else "Acceptance Rate", fig_rate), card(f"{top_cohort} Largest - {int(top_cohort_val)} Applicants", fig_cohort)]), html.Div(style={'display':'flex','gap':'14px','flexWrap':'wrap'}, children=[card("Top Sectors Vary by Cohort", fig_sec_cohort), card("Applicant Type Evolution", fig_type_cohort)])]

def run_dash(): app.run(port=8050, host='0.0.0.0', debug=False, use_reloader=False)
thread = threading.Thread(target=run_dash); thread.daemon=True; thread.start()
time.sleep(2); print(ngrok.connect(8050))

<IPython.core.display.Javascript object>

NgrokTunnel: "https://pulmonary-elope-sanctuary.ngrok-free.dev" -> "http://localhost:8050"
